In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Load Dataset**

In [2]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("ag_news")

# Split into train and test
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]
test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

**Embedding Using Bert**

In [3]:
from sentence_transformers import SentenceTransformer

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Embed train and test texts
X_train = embedder.encode(train_texts, convert_to_tensor=False)
X_test = embedder.encode(test_texts, convert_to_tensor=False)

2025-05-06 09:52:07.180202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746525127.355586      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746525127.408133      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3750 [00:00<?, ?it/s]

Batches:   0%|          | 0/238 [00:00<?, ?it/s]

In [4]:
print(X_train[:5])

[[ 0.00743863  0.0285624   0.04109555 ... -0.13397595 -0.0901915
   0.06469314]
 [-0.00054744 -0.1444023  -0.0526396  ... -0.09893098  0.01725998
   0.06443578]
 [-0.0055956  -0.02277451  0.07513669 ... -0.07831726  0.00378908
   0.06476948]
 [ 0.0079434  -0.0802167   0.07207926 ...  0.04463271  0.0331801
   0.0257829 ]
 [-0.06110712 -0.04299483  0.07979128 ... -0.073373    0.01544186
   0.0661686 ]]


**Saving Embeddings**

In [5]:
import numpy as np

# Save embeddings
np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)

# Convert labels to numpy and save
np.save("y_train.npy", np.array(train_labels))
np.save("y_test.npy", np.array(test_labels))

print("Embeddings saved")

Embeddings saved


**SVM**

In [6]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

svm_model = SVC(kernel="linear")
svm_model.fit(X_train, train_labels)
svm_preds = svm_model.predict(X_test)

print("SVM Classification Report:\n", classification_report(test_labels, svm_preds))

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.89      0.90      1900
           1       0.96      0.98      0.97      1900
           2       0.86      0.86      0.86      1900
           3       0.87      0.87      0.87      1900

    accuracy                           0.90      7600
   macro avg       0.90      0.90      0.90      7600
weighted avg       0.90      0.90      0.90      7600



**Save Model**

In [7]:
import joblib

# Save the trained SVM model
joblib.dump(svm_model, "svm_model.pkl")

print("Model Saved")

Model Saved


**RandomForest**

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100)
rf_model.fit(X_train, train_labels)
rf_preds = rf_model.predict(X_test)

print("Random Forest Classification Report:\n", classification_report(test_labels, rf_preds))

Random Forest Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.88      0.89      1900
           1       0.94      0.97      0.96      1900
           2       0.86      0.85      0.85      1900
           3       0.86      0.86      0.86      1900

    accuracy                           0.89      7600
   macro avg       0.89      0.89      0.89      7600
weighted avg       0.89      0.89      0.89      7600



**Save Model**

In [9]:
import joblib

# Save the trained SVM model
joblib.dump(rf_model, "rf_model.pkl")

print("Model Saved")

Model Saved


**FFNN**

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder

# If not done already
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(train_labels, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(test_labels, dtype=torch.long)

# Create dataloaders
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64)

# Define the model
class FFNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(FFNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        out = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(out)

# Initialize model
input_size = X_train.shape[1]  # 384 if using MiniLM
model = FFNN(input_size, hidden_size=128, num_classes=4)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
for epoch in range(5):  # Increase for better results
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluation
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
print(f"Accuracy: {100 * correct / total:.2f}%")

Epoch 1, Loss: 0.2196
Epoch 2, Loss: 0.2355
Epoch 3, Loss: 0.1522
Epoch 4, Loss: 0.3463
Epoch 5, Loss: 0.2740
Accuracy: 91.04%
